# OpenSky Collector — Step by Step

This notebook explains the collector and runs it.

**Architecture:**
```
OpenSky Network API  →  get_departures() / get_arrivals()  →  build_document()  →  MongoDB Atlas (airline_landing.opensky_raw)
```

**Important:** The collector must run locally — OpenSky OAuth2 auth (`auth.opensky-network.org`) is only reachable from the Mac (blocked on external VMs). See ADR 004 / ADR 005.

**Prerequisite:** `.env` at the **project root** (`airline-data-platform/.env`) with:
```
OPENSKY_CLIENT_ID=...
OPENSKY_CLIENT_SECRET=...
MONGO_URI_RW=mongodb+srv://airline-collector-rw:<password>@mongo-mk1.ptb1k2b.mongodb.net/airline_landing
MONGO_DB=airline_landing
```
This collector writes data, so it uses `MONGO_URI_RW` (the `airline-collector-rw` user with write access) via `from_env(write=True)`.
Credentials: `.env` is gitignored, never commit.

## Step 1 — Imports and path setup

The collector lives in `collectors/opensky_collector.py`.  
The OpenSky client lives in `opensky_api/client.py`.  
The MongoDB connector lives in `db/mongo/connector.py`.  
To let Python find these modules, the current directory (`03-data-collection/`) must be on the search path.

In [ ]:
import sys
from pathlib import Path

notebook_dir = Path(".").resolve()
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

print(f"Working directory: {notebook_dir}")

## Step 2 — Configure the time window

The OpenSky Flights API returns completed flights in a time window (`begin` / `end` as Unix timestamps).  
Default look-back is 24 hours. The window can span at most 7 days.

| Parameter | Description |
|---|---|
| `begin` | Window start (Unix timestamp) |
| `end` | Window end (Unix timestamp, typically `now`) |
| `airport` | ICAO code, e.g. `EDDB` (Berlin Brandenburg) |

In [ ]:
import time
from datetime import datetime, timezone

AIRPORT = "EDDB"
LOOK_BACK_HOURS = 24

end = int(time.time())
begin = end - LOOK_BACK_HOURS * 3600

print(f"Airport:  {AIRPORT}")
print(f"Begin:    {begin}  ({datetime.fromtimestamp(begin, tz=timezone.utc).isoformat()})")
print(f"End:      {end}  ({datetime.fromtimestamp(end, tz=timezone.utc).isoformat()})")
print(f"Window:   {LOOK_BACK_HOURS} hours")

## Step 3 — Query the API (mock mode)

First with `use_mock=True` — no login, no network connection needed.  
Mock data in `opensky_api/mock_data.py` mirrors the real API response structure.

We query **departures** and **arrivals** for `EDDB` — one separate API call each.

In [ ]:
from opensky_api.client import OpenSkyClient

client_mock = OpenSkyClient(use_mock=True)

departures = client_mock.get_departures(AIRPORT, begin, end)
arrivals   = client_mock.get_arrivals(AIRPORT, begin, end)

print(f"Departures: {len(departures)} flights")
print(f"Arrivals:   {len(arrivals)} flights")
print()
print("First departure (raw):")
departures[0] if departures else print("(no data)")

## Step 4 — Inspect flights as a DataFrame

The API returns one dict per flight. Key fields:

| Field | Description |
|---|---|
| `icao24` | Aircraft transponder address |
| `callsign` | Callsign (e.g. `EWG1R`) — may have trailing spaces |
| `firstSeen` | Unix timestamp departure |
| `lastSeen` | Unix timestamp arrival |
| `estDepartureAirport` | ICAO code departure airport |
| `estArrivalAirport` | ICAO code destination airport |
| `estDepartureAirportHorizDistance` | Horizontal distance to departure airport (m) |

In [ ]:
import pandas as pd

df_dep = pd.DataFrame(departures)
df_arr = pd.DataFrame(arrivals)

print(f"Departures — Shape: {df_dep.shape}")

show = [c for c in ['icao24', 'callsign', 'estDepartureAirport', 'estArrivalAirport', 'firstSeen', 'lastSeen'] if c in df_dep.columns]
df_dep[show]

## Step 5 — Build the document

`build_document()` wraps the raw flight list in the MongoDB format.  
The `flights` array is kept **unchanged** — no transformation, no flattening.  
This is intentional: MongoDB is the raw landing zone; the ETL layer normalises later.

In [ ]:
from collectors.opensky_collector import build_document

doc_dep = build_document("departures", AIRPORT, begin, end, departures)
doc_arr = build_document("arrivals",   AIRPORT, begin, end, arrivals)

print("Departures document (excluding flights array):")
for k, v in doc_dep.items():
    if k != "flights":
        print(f"  {k}: {v}")
print(f"  flights: [{len(doc_dep['flights'])} flight dicts]")

## Step 6 — Live API (optional)

Set `use_mock=False` for a live fetch. Requires `.env` with valid credentials.

```python
client_live = OpenSkyClient(use_mock=False)
departures_live = client_live.get_departures(AIRPORT, begin, end)
```

The OAuth2 token URL is `https://auth.opensky-network.org/...` — external VMs cannot reach it,
so this collector runs **locally only** (see ADR 004).

## Step 7 — Connect to MongoDB

`from_env(write=True)` reads `MONGO_URI_RW` and `MONGO_DB` from the `.env` file at the project root — the collector needs write access.  
MongoDB runs on **MongoDB Atlas** (`mongo-mk1`, eu-central-1) — connection via SRV URI.

In [ ]:
from db.mongo.connector import from_env

db = from_env(write=True).connect()
print("Connected.")

## Step 8 — Write snapshots to MongoDB

`insert_raw()` calls `collection.insert_one(doc)` internally.  
Each collector run produces **two documents**: one for departures, one for arrivals.

In [ ]:
id_dep = db.insert_raw("opensky_raw", doc_dep)
id_arr = db.insert_raw("opensky_raw", doc_arr)

print(f"Departures stored: _id={id_dep}")
print(f"Arrivals   stored: _id={id_arr}")

total = db.collection("opensky_raw").count_documents({})
print(f"\nTotal documents in opensky_raw: {total}")

## Step 9 — Read back and verify

Quick verification: fetch the last two documents from `opensky_raw` and display metadata.

In [ ]:
col = db.collection("opensky_raw")

print("Letzte 2 Dokumente in opensky_raw:")
for doc in col.find({}, {"flights": 0}).sort("collected_at", -1).limit(2):
    doc["_id"] = str(doc["_id"])
    print(doc)

## Step 10 — Close connection

In production the collector runs as a script (no notebook needed):  
```bash
cd 03-data-collection
python collectors/opensky_collector.py --hours 24
python collectors/opensky_collector.py --mock   # no credentials needed
```

In [ ]:
db.close()
print("Connection closed.")